# CV Masterclass 11: Edge Deployment & TensorRT

Welcome to Notebook 11, the final module in the Computer Vision Masterclass.

A 100-layer ResNet trained in PyTorch on an A100 GPU is mathematically flawless. But if you try to run it on a small drone's Raspberry Pi or an Nvidia Jetson Nano, you will get 2 Frames Per Second (FPS) and the battery will drain in 4 minutes.

Producing an AI model is useless if you cannot deploy it to the Edge.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"When converting a PyTorch FP32 model to an optimized INT8 TensorRT engine, you must provide a 'Calibration Dataset' of ~1,000 images before it converts. Why? What exact statistical properties is the compiler searching for in those 1,000 images, and what happens to your YOLO's bounding boxes if you perform Quantization without them?"*

> *"A student quantizes a ResNet-50 using PTQ with INT8 and sees only 0.3% accuracy loss. She then quantizes a transformer-based ViT-B/16 with the same PTQ pipeline and sees 8% accuracy loss, even with a 1000-image calibration set. Why are transformer architectures specifically more sensitive to post-training INT8 quantization than ResNets, and what two properties of the attention mechanism's activation distribution make KL-divergence calibration insufficient without architectural modifications?"*

---

## Table of Contents
1. **The Deployment Pipeline:** PyTorch -> ONNX -> TensorRT.
2. **Graph Optimization:** Layer fusion and memory pruning.
3. **Knowledge Distillation:** Compressing knowledge from Teacher to Student.
4. **Pruning:** Structured vs Unstructured sparsification.
5. **FP32 vs INT8:** The physics of Quantization.
6. **Calibration (Kullback-Leibler Divergence):** Squeezing infinity into 256 bins.
7. **Quantization-Aware Training (QAT):** Simulating INT8 during backprop.
8. **Cross-Platform Deployment:** CoreML, TFLite, and OpenVINO.


## 1. The Deployment Pipeline

`torch.save(model.state_dict())` is strictly for Python. You cannot reliably run Python on Edge devices (it requires a massive interpreter and GIL overhead).

1.  **ONNX (Open Neural Network Exchange):** We export the PyTorch model to an ONNX file. ONNX throws away all your Python classes and `def forward()` loops. It tracks the exact mathematical sequence of tensors and compiles a static C++ computation graph.
2.  **TensorRT (TRT):** ONNX is agnostic. Nvidia TensorRT takes the ONNX graph and physically compiles it specifically for the exact silicon architecture of the target device (e.g., maximizing the exact number of Tensor Cores available on a Jetson Orin).


## 2. Graph Optimization & Layer Fusion

In PyTorch, a standard layer is: `Conv2d -> BatchNorm2d -> ReLU`.

During inference, reading and writing to GPU memory across three separate algorithmic steps is slow. 

TensorRT physically "fuses" these layers into a single mathematical step. It absorbs the Batch Norm scaling directly into the Convolution Weights (math allows this because they are both linear operations), and appends the ReLU threshold to the kernel output. 

**Result:** 1 memory read, 1 memory write. The model becomes twice as fast before we even touch data types.


In [ ]:
import torch
import torch.nn as nn

# -----------------------------------------------------
# IMPLEMENTATION: Mathematical Batch Norm Fusion
# -----------------------------------------------------

def fuse_conv_and_bn(conv, bn):
    '''
    Simulates what TensorRT does natively.
    Absorbs BatchNorm parameters directly into the Conv2d weights.
    '''
    bn_mean = bn.running_mean
    bn_var = bn.running_var
    bn_eps = bn.eps
    bn_weight = bn.weight # Gamma
    bn_bias = bn.bias     # Beta

    conv_weight = conv.weight
    conv_bias = conv.bias if conv.bias is not None else torch.zeros_like(bn_mean)

    # 3. The Fusion Math
    bn_std = torch.sqrt(bn_var + bn_eps)
    multiplier = bn_weight / bn_std

    # Apply to Conv Weights
    fused_weight = conv_weight * multiplier.view(-1, 1, 1, 1)
    # Apply to Conv Bias
    fused_bias = (conv_bias - bn_mean) * multiplier + bn_bias

    # Create a new, pure Conv layer (Batch Norm is deleted!)
    fused_conv = nn.Conv2d(conv.in_channels, out_channels=conv.out_channels,
                           kernel_size=conv.kernel_size, stride=conv.stride,
                           padding=conv.padding, bias=True)
                           
    fused_conv.weight = nn.Parameter(fused_weight)
    fused_conv.bias = nn.Parameter(fused_bias)
    
    return fused_conv

# TEST IT
conv = nn.Conv2d(3, 16, 3, padding=1)
bn = nn.BatchNorm2d(16)
# Simulate a forward pass so BN tracks running stats
dummy = torch.randn(1, 3, 32, 32)
bn(conv(dummy))

fused_layer = fuse_conv_and_bn(conv, bn)
print(f"Fused Layer Bias Shape: {fused_layer.bias.shape} (BN absorbed)")


### COMMON PITFALLS
*   **Assuming dynamic control flow will export easily:** If your model uses python `if x.sum() > 0:` loops inside `forward()`, ONNX export will either break entirely or statically bake the conditional evaluated during tracing, ruining your logic in deployment. Use torch operations exclusively.


## 3. Knowledge Distillation

A ResNet-152 (60M params, 11 GFLOPs) achieves 78% on ImageNet. A MobileNetV3 (5M params, 0.2 GFLOPs) achieves 67% ImageNet from scratch. Can we transfer the massive model's latent knowledge into the tiny model?

**Knowledge Distillation** (Hinton 2015) trains the Student on TWO distinct losses simultaneously:
1.  **Hard Loss:** Standard Cross-Entropy against the ground truth one-hot labels.
2.  **Soft Loss:** Kullback-Leibler (KL) divergence between the *Student's* softmax predictions and the *Teacher's* softmax predictions, both computed at an elevated Temperature $T > 1$.

**Why Temperature?**
At $T=1$, the massive Teacher model predicts `[0.98, 0.01, 0.01, ...]`. This is basically a hard one-hot label.
At $T=4$, the exponent scaling softens the distribution to `[0.6, 0.2, 0.2, ...]`. 
This directly reveals the Teacher's inter-class similarity beliefs: class "Cat" and "Tiger" have a high soft similarity because the Teacher was nearly confused between them deep in its feature representations. This "dark knowledge" is vastly richer information than just "the answer is cat," allowing the tiny MobileNet to replicate the decision boundaries without needing 60M parameters.


In [ ]:
import torch.nn.functional as F
import torchvision.models as models

# -----------------------------------------------------
# IMPLEMENTATION: Knowledge Distillation Training Step
# -----------------------------------------------------

def distillation_loss_step(student_logits, teacher_logits, true_labels, T=4.0, alpha=0.7):
    # 1. Hard Cross-Entropy Loss
    hard_loss = F.cross_entropy(student_logits, true_labels)
    
    # 2. Soft KL Divergence Loss at Temperature T
    soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_teacher = F.softmax(teacher_logits / T, dim=1)
    
    # KLDiv requires log probabilities for input, and standard probabilities for target
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (T ** 2)
    
    # Combine
    return (1.0 - alpha) * hard_loss + alpha * soft_loss

# TEST IT
# Simulate a training loop batch
teacher = models.resnet50()
teacher.eval() # Teacher must NOT learn

# Student
student = models.mobilenet_v3_small()
student.train()

inputs = torch.randn(2, 3, 224, 224)
labels = torch.tensor([10, 42]) # Dummy ImageNet labels

with torch.no_grad():
    t_logits = teacher(inputs)

s_logits = student(inputs)
loss = distillation_loss_step(s_logits, t_logits, labels, T=4.0, alpha=0.7)

print(f"Distillation Loss Step Completed: {loss.item():.4f}")
print("Distilling ResNet-50 into MobileNetV3 typically yields MobileNet models recovering up to 72%+ ImageNet accuracy!")


### COMMON PITFALLS
*   **Leaving the Teacher in `.train()` mode:** If you do not call `teacher.eval()`, the Teacher's Batch Norm layers will dynamically update their running statistics based on the batch, mathematically shifting its internal distribution and causing catastrophic divergence in the Student.


## 4. Structured vs Unstructured Pruning

Trained networks have millions of redundant weights (values naturally close to 0). We can set them to exactly $0.0$ without losing accuracy—this is called **Pruning**.

*   **Unstructured Pruning:** Zeroing out individual individual weights scattered across the matrix. A 90% sparse weight matrix has 90% zeros. However, modern GPU hardware (like Nvidia Tensor Cores) cannot exploit this random sparsity—they expect dense matrices for GEMM (Matrix Multiplication). Result: smaller file size on disk, but absolutely NO latency improvement.
*   **Structured Pruning:** Zeroing out entire mathematical *filters* or *channels*. If a Conv2d layer has 128 filters and we prune the 32 lowest-magnitude (L1-norm) filters, the layer violently mutates into a (96)-filter layer organically. Result: actual physical latency improvement, smaller model, small accuracy loss.

After pruning, we typically fine-tune the model for $\sim 5$ epochs to recover the broken pathways. Looking at a Pareto curve of Pruning Ratio vs Accuracy, pruning 30% of filters often yields $0\%$ accuracy loss but a massive speedup!


In [ ]:
import torch.nn.utils.prune as prune

# -----------------------------------------------------
# IMPLEMENTATION: Structured L1-Norm Pruning
# -----------------------------------------------------

# Let's prune a convolutional layer
test_conv = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)

# Look at original parameters
print(f"Original Weight Shape: {test_conv.weight.shape}")

# Calculate L1-Norm of all 128 filters, and zero out the worst 30% entire filters along dimension 0
prune.ln_structured(test_conv, name="weight", amount=0.3, n=1, dim=0)

# PyTorch Prune module operates via applying a mathematical mask dynamically
masked_weight = test_conv.weight
zeros_count = (masked_weight == 0).sum().item()
total_params = masked_weight.numel()

print(f"After Structured Pruning: {zeros_count} out of {total_params} parameters are physically 0.0")

# To make it permanent and remove the dynamic mask overlay for ONNX export:
prune.remove(test_conv, 'weight')


### COMMON PITFALLS
*   **Exporting without removing masks:** PyTorch's pruning library uses hooks to dynamically apply a 0-mask during the forward pass. If you do not explicitly run `prune.remove()`, exporting to ONNX will include the masking arithmetic dynamically, actually making the model *slower* instead of pruning it.


## 5. FP32 vs INT8 (Quantization Physics)

PyTorch models use 32-bit Floating Point numbers (`FP32`). These represent massive ranges with infinite decimals.

Edge hardware runs incredibly fast on 8-bit Integers (`INT8`). An `INT8` can only represent exactly 256 physical distinct numbers (e.g., $-128$ to $+127$).

If a Convolution outputs the activation value `3.14159`, how do we map that into the `[-128, +127]` integer bin? We scale it.

## 6. Calibration & Kullback-Leibler Divergence

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does TensorRT demand a "Calibration Dataset" before compiling INT8?*

**A:** 
To map an `FP32` space into exactly 256 an `INT8` bins, TensorRT needs to know the physical maximum value the network outputs. 
But Neural Networks have **Outliers**. Say 99% of your activations are between `-2.0` and `2.0`, but occasionally one neuron screams `90.0`.

If TensorRT naively sets the boundary at `90.0 -> 127`, then the entire critical range of `[-2.0, 2.0]` (where the actual object features live) is compressed into a single INT8 bin (e.g., `Bin 0`). All mathematical detail is utterly destroyed. Your bounding boxes will randomize or disappear.

**The Calibration Solution:**
You feed TensorRT 1,000 images representing the real-world factory. It physically tracks the exact statistical distribution of every single neuron. It uses **Kullback-Leibler (KL) Divergence** to find the optimal "Clip Value". It might decide to set the map boundary at `3.0 -> 127`, preserving the vast majority of the data beautifully, and happily truncating the `90.0` outlier down to `127` as an acceptable loss.


## 7. Quantization-Aware Training (QAT)

**The Problem with Post-Training Quantization (PTQ):**
PTQ's calibration step (KL Divergence) minimizes quantization error locally layer-by-layer. But across a 100-layer network, microscopic `INT8` rounding errors compound catastrophically. The calibration step cannot "backpropagate" to tell earlier layers to adjust their weights to fix the compounding errors.

**Quantization-Aware Training (QAT):**
QAT explicitly inserts "Fake Quantization" nodes into the PyTorch training graph.
- *Forward Pass:* Weights & Activations are clipped and quantized to INT8, then dequantized back to FP32, forcing the math to physically simulate INT8 arithmetic.
- *Backward Pass:* Because a step-function isn't differentiable, QAT uses a **Straight-Through Estimator (STE)**, which mathematically lies and passes gradients straight through the fake-quantize node as if it were an identity function.

The model explicitly LEARNS to be quantization-resilient. Weights that generate fatal rounding errors get organically regularized away.

| Strategy | FP32 Target | PTQ INT8 | QAT INT8 | Needs Calibration Phase |
|---|---|---|---|---|
| **Accuracy** | Baseline (78%) | 75.1% | **77.6%** | Yes (PTQ) vs Embedded (QAT) |
| **Latency/Speedup**| 1.0x | ~3.5x | ~3.5x | N/A |
| **Model Size** | 100% | 25% | 25% | N/A |


In [ ]:
import torch.quantization

# -----------------------------------------------------
# IMPLEMENTATION: QAT Graph Preparation
# -----------------------------------------------------
class DummyConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        # QAT requires manual insertion of Quantize and DeQuantize stubs
        self.quant = torch.quantization.QuantStub()
        self.conv = nn.Conv2d(3, 16, 3)
        self.relu = nn.ReLU()
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.conv(x)
        x = self.relu(x)
        x = self.dequant(x)
        return x

qat_model = DummyConvNet()

# 1. Attach QAT configuration mapping mappings (e.g. fbgemm or qnnpack)
qat_model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')

# 2. Fuse layers manually before QAT tracing so simulation is accurate
# (Conv + ReLU)
torch.quantization.fuse_modules(qat_model, [['conv', 'relu']], inplace=True)

# 3. Explicitly prepare the model for QAT (inserts fake quantization observer modules)
torch.quantization.prepare_qat(qat_model, inplace=True)

# You would train the model normally for 5-10 epochs here.

# 4. Convert fully to integer mode after QAT simulates physical convergence
torch.quantization.convert(qat_model, inplace=True)
print("QAT Simulation graph fully built and mapped to INT8 successfully.")


### COMMON PITFALLS
*   **Assuming QAT applies globally without fused modules:** If you run QAT without physically fusing Conv+BatchNorm+ReLU beforehand in PyTorch, your fake-quantize trackers will estimate boundaries on intermediate layers that physically will not exist in TensorRT. Your gradients will optimize for entirely incorrect topologies.


## 8. Cross-Platform Deployment (Beyond TensorRT)

TensorRT is strictly for Nvidia hardware. Let's build a decision matrix for deploying everywhere else.

1.  **CoreML (Apple Silicon / iOS):**
    *   Export via `coremltools`. 
    *   *Pipeline:* PyTorch $\to$ TorchScript/ONNX $\to$ CoreML `.mlpackage`.
    *   *Note:* The ANE (Apple Neural Engine) only accelerates very specific spatial tensor shapes (typically standard NCHW convolutions). Some transformer architectures will silently fall back to the generic CPU.
2.  **TFLite (Android / Mobile Edge):**
    *   Export requires converting the mathematical graph format explicitly to TensorFlow primitives.
    *   *Pipeline:* PyTorch $\to$ ONNX $\to$ `tf.saved_model` $\to$ TFLite.
    *   *Note:* TFLite supports hardware *per-channel* quantization, which allows different integer scaling values per individual convolution filter, significantly preserving accuracy over blunt per-tensor quantization.
3.  **OpenVINO (Intel CPU / Camera Edge):**
    *   Essential for industrial cameras mounted to robotic arms connected to cheap generic Intel hardware.
    *   *Pipeline:* PyTorch $\to$ ONNX $\to$ OpenVINO Model Optimizer.

### Hardware Decision Matrix

| Hardware Target | Primary Accelerator Engine | Optimal Deployment Pipeline | Hardware INT8 Support |
| :--- | :--- | :--- | :--- |
| **Nvidia Jetson / PCIe GPUs** | Tensor Cores / CUDA | PyTorch $\to$ ONNX $\to$ TensorRT | Exceptional (PTQ/QAT) |
| **iOS / Apple Silicon** | Apple Neural Engine (ANE) | PyTorch $\to$ CoreML (`coremltools`) | FP16 Native, specific INT8 |
| **Android / Arm Cortex** | NNAPI / Hexagon DSP | PyTorch $\to$ ONNX $\to$ TF $\to$ TFLite | Fully defined PTQ |
| **Generic x86 / Industrial CPU** | Intel AVX-512 / VNNI | PyTorch $\to$ ONNX $\to$ OpenVINO | CPU Vectorized INT8 |


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: The Universal ONNX Entrypoint
# -----------------------------------------------------
# Regardless of what Edge device you deploy to, ONNX is usually the intermediate bridge.

dummy_input = torch.randn(1, 3, 224, 224, device='cpu')
vision_model = models.resnet18(pretrained=True).eval()

# Exporting requires establishing static input/output node names for C++ APIs
torch.onnx.export(
    vision_model,                   # model being run
    dummy_input,                    # dummy input tensor required to trace the graph
    "production_resnet18.onnx",     # where to save the model
    export_params=True,             # store the trained parameter weights
    opset_version=14,               # standard opset compatible with modern TRT/TFLite
    do_constant_folding=True,       # fold constant nodes for optimization
    input_names = ['input_image'],  # input name for inference API
    output_names = ['logits'],      # output name for inference API
    dynamic_axes={'input_image': {0: 'batch_size'},    # allow variable sized batches
                  'logits': {0: 'batch_size'}}
)
print("ONNX universally exported. Ready for TRT/OpenVINO compilers.")



---
# Computer Vision: Masterclass Complete 🎓

You have successfully navigated from the physics of discrete $3\times3$ convolutions, through the identity mechanics of ResNet, the geometric scaling of U-Net, the infinite dimensional spaces of Vision Transformers, the contrastive dynamics of Self-Supervised Learning, the generative realities of Diffusion models, and finally, compiled it all onto physical Edge hardware.

The 11-module CV theoretical foundation is mathematically secure.
